# Soil Sustainability Analysis

**Date:** 2026-04-20
**Data Sources:** 
- `NC_soil_crop_data_summary_EPSG4326_2026-04-01.csv` - Soil and crop summary data
- `NC_soil_crop_data_EPSG4326_2026-04-01.csv` - Detailed horizon data

## Summary

This notebook analyzes key soil health indicators for agricultural fields in North Carolina:
- **Organic Matter (OM)** - critical for soil structure, water retention, and nutrient supply
- **pH** - affects nutrient availability and microbial activity
- **Cation Exchange Capacity (CEC)** - indicates soil's ability to retain nutrients

We calculate a composite soil health score to identify fields with the best sustainability potential.
Additionally, we assess **erosion risk** and **carbon storage potential**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 150

DATA_DIR = Path("/workspaces/my-farm-advisor/data/workspace")
PLOTS_DIR = DATA_DIR / "plots"
OUTPUT_DIR = DATA_DIR / "output"

## 1. Data Loading

In [ ]:
df = pd.read_csv(DATA_DIR / "NC_soil_crop_data_summary_EPSG4326_2026-04-01.csv")
print(f"Loaded {len(df)} fields")
df.head()

## 2. Filter Key Soil Health Indicators

In [ ]:
key_indicators = ['avg_om_pct', 'avg_ph', 'avg_cec']
df_health = df[['field_id', 'dominant_soil', 'drainage_class'] + key_indicators].copy()
df_health = df_health.dropna(subset=key_indicators)
print(f"Fields with complete key indicator data: {len(df_health)}")
df_health.head(10)

## 3. Descriptive Statistics

In [ ]:
df_health[key_indicators].describe()

## 4. Distribution Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

indicators_labels = {
    'avg_om_pct': 'Organic Matter (%)',
    'avg_ph': 'pH',
    'avg_cec': 'CEC (meq/100g)'
}

for ax, col in zip(axes, key_indicators):
    ax.hist(df_health[col], bins=10, edgecolor='black', alpha=0.7, color='forestgreen')
    ax.set_xlabel(indicators_labels[col])
    ax.set_ylabel('Frequency')
    ax.axvline(df_health[col].mean(), color='red', linestyle='--', label='Mean')
    ax.legend()

plt.suptitle('Distribution of Key Soil Health Indicators', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'soil_health_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Analysis

In [ ]:
corr_matrix = df_health[key_indicators].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', center=0, fmt='.2f',
            xticklabels=['OM (%)', 'pH', 'CEC'],
            yticklabels=['OM (%)', 'pH', 'CEC'])
plt.title('Correlation Between Soil Health Indicators')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'soil_health_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Field-Level Soil Health Score

In [ ]:
df_score = df_health.copy()

df_score['om_score'] = (df_score['avg_om_pct'] - df_score['avg_om_pct'].min()) / (df_score['avg_om_pct'].max() - df_score['avg_om_pct'].min())
df_score['ph_score'] = 1 - np.abs(df_score['avg_ph'] - 6.5) / np.abs(df_score['avg_ph'] - 6.5).max()
df_score['cec_score'] = (df_score['avg_cec'] - df_score['avg_cec'].min()) / (df_score['avg_cec'].max() - df_score['avg_cec'].min())

df_score['soil_health_score'] = (df_score['om_score'] * 0.4 + 
                                   df_score['ph_score'] * 0.3 + 
                                   df_score['cec_score'] * 0.3)

df_score = df_score.sort_values('soil_health_score', ascending=False)
df_score

## 7. Top Performing Fields

In [ ]:
top_fields = df_score.head(10)[['field_id', 'dominant_soil', 'drainage_class', 
                                   'avg_om_pct', 'avg_ph', 'avg_cec', 'soil_health_score']]
print("Top 10 Fields by Soil Health Score:\n")
top_fields

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(df_score)))[::-1]
bars = ax.barh(df_score['field_id'], df_score['soil_health_score'], color=colors)
ax.set_xlabel('Soil Health Score')
ax.set_ylabel('Field ID')
ax.set_title('Soil Health Score by Field (Higher is Better)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'soil_health_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Findings

In [ ]:
best_field = df_score.iloc[0]
print("=== SOIL SUSTAINABILITY SUMMARY ===\n")
print(f"Best performing field: {best_field['field_id']}")
print(f"  - Dominant soil: {best_field['dominant_soil']}")
print(f"  - Drainage: {best_field['drainage_class']}")
print(f"  - OM: {best_field['avg_om_pct']:.2f}%")
print(f"  - pH: {best_field['avg_ph']:.2f}")
print(f"  - CEC: {best_field['avg_cec']:.2f} meq/100g")
print(f"  - Health Score: {best_field['soil_health_score']:.3f}\n")

print(f"Average across all fields:")
print(f"  - OM: {df_health['avg_om_pct'].mean():.2f}%")
print(f"  - pH: {df_health['avg_ph'].mean():.2f}")
print(f"  - CEC: {df_health['avg_cec'].mean():.2f} meq/100g")

## 9. Erosion Risk Assessment

Erosion risk is estimated using proxy indicators from NRCS data:
- **Sand content**: Higher sand % = lower cohesion = higher erosion risk
- **Slope gradient**: Extracted from soil mapunit names (e.g., "8 to 15 percent slopes")
- **Drainage class**: Poorly drained soils can have surface runoff issues

Formula: `Erosion Risk = Sand Risk (40%) + Slope Risk (60%)`

In [ ]:
def extract_slope_pct(muname):
    if pd.isna(muname):
        return 0
    match = re.search(r'(\d+)\s*to\s*(\d+)\s*percent', str(muname))
    if match:
        low, high = int(match.group(1)), int(match.group(2))
        return (low + high) / 2
    return 0

df['slope_pct'] = df['dominant_muname'].apply(extract_slope_pct)

df_erosion = df[['field_id', 'dominant_soil', 'drainage_class', 'avg_sand_pct', 'slope_pct']].copy()
df_erosion = df_erosion.dropna(subset=['avg_sand_pct'])

df_erosion['sand_risk'] = (df_erosion['avg_sand_pct'] - df_erosion['avg_sand_pct'].min()) / (df_erosion['avg_sand_pct'].max() - df_erosion['avg_sand_pct'].min())
df_erosion['slope_risk'] = (df_erosion['slope_pct'] - df_erosion['slope_pct'].min()) / (df_erosion['slope_pct'].max() - df_erosion['slope_pct'].min())
df_erosion['erosion_risk_score'] = df_erosion['sand_risk'] * 0.4 + df_erosion['slope_risk'] * 0.6

df_erosion = df_erosion.sort_values('erosion_risk_score', ascending=False)
print("Fields Ranked by Erosion Risk (Higher = More Risk):\n")
df_erosion[['field_id', 'dominant_soil', 'slope_pct', 'avg_sand_pct', 'erosion_risk_score']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.8, len(df_erosion)))
ax.barh(df_erosion['field_id'], df_erosion['erosion_risk_score'], color=colors)
ax.set_xlabel('Erosion Risk Score')
ax.set_ylabel('Field ID')
ax.set_title('Erosion Risk by Field (Higher = More Risk)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'erosion_risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Carbon Storage Potential

Carbon storage potential is estimated using:
- **Topsoil depth**: From horizon data (hzdepb_r - hzdept_r for the top horizon)
- **Organic Matter %**: OM is the primary carbon source
- **Bulk density**: Affects the mass of carbon that can be stored per unit volume

Formula: `Carbon Storage = OM (%) * Bulk Density * Topsoil Depth (cm)`

In [ ]:
df_detailed = pd.read_csv(DATA_DIR / "NC_soil_crop_data_EPSG4326_2026-04-01.csv")
df_tophz = df_detailed[df_detailed['hzdept_r'] == 0].groupby('field_id').first().reset_index()
df_tophz = df_tophz[['field_id', 'hzdepb_r', 'om_r', 'dbthirdbar_r']].copy()
df_tophz.columns = ['field_id', 'topsoil_depth_cm', 'om_pct', 'bulk_density']

df_carbon = df[['field_id', 'dominant_soil', 'drainage_class', 'avg_om_pct', 'avg_bulk_density']].merge(
    df_tophz[['field_id', 'topsoil_depth_cm']], on='field_id', how='left'
)
df_carbon = df_carbon.dropna(subset=['topsoil_depth_cm', 'avg_om_pct', 'avg_bulk_density'])

df_carbon['carbon_storage'] = df_carbon['avg_om_pct'] * df_carbon['avg_bulk_density'] * df_carbon['topsoil_depth_cm']
df_carbon = df_carbon.sort_values('carbon_storage', ascending=False)
print("Fields Ranked by Carbon Storage Potential:\n")
df_carbon[['field_id', 'dominant_soil', 'avg_om_pct', 'avg_bulk_density', 'topsoil_depth_cm', 'carbon_storage']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Greens(np.linspace(0.3, 0.8, len(df_carbon)))
ax.barh(df_carbon['field_id'], df_carbon['carbon_storage'], color=colors)
ax.set_xlabel('Carbon Storage Potential')
ax.set_ylabel('Field ID')
ax.set_title('Carbon Storage Potential by Field (Higher = Better)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'carbon_storage_potential.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Combined Sustainability Dashboard

Combining all metrics into a single sustainability index:
- Soil Health Score (40%): Based on OM, pH, CEC
- Erosion Resistance (30%): Inverse of erosion risk
- Carbon Storage (30%): Based on OM, bulk density, topsoil depth

In [ ]:
df_combined = df_score[['field_id', 'soil_health_score']].merge(
    df_erosion[['field_id', 'erosion_risk_score']], on='field_id'
).merge(
    df_carbon[['field_id', 'carbon_storage']], on='field_id'
)
df_combined = df_combined.dropna()

df_combined['health_norm'] = (df_combined['soil_health_score'] - df_combined['soil_health_score'].min()) / (df_combined['soil_health_score'].max() - df_combined['soil_health_score'].min())
df_combined['erosion_norm'] = (df_combined['erosion_risk_score'] - df_combined['erosion_risk_score'].min()) / (df_combined['erosion_risk_score'].max() - df_combined['erosion_risk_score'].min())
df_combined['carbon_norm'] = (df_combined['carbon_storage'] - df_combined['carbon_storage'].min()) / (df_combined['carbon_storage'].max() - df_combined['carbon_storage'].min())
df_combined['sustainability_index'] = (df_combined['health_norm'] * 0.4 + 
                                    (1 - df_combined['erosion_norm']) * 0.3 + 
                                    df_combined['carbon_norm'] * 0.3)

df_combined = df_combined.sort_values('sustainability_index', ascending=False)
print("Fields Ranked by Overall Sustainability Index:\n")
df_combined[['field_id', 'soil_health_score', 'erosion_risk_score', 'carbon_storage', 'sustainability_index']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(df_combined)))[::-1]
ax.barh(df_combined['field_id'], df_combined['sustainability_index'], color=colors)
ax.set_xlabel('Sustainability Index')
ax.set_ylabel('Field ID')
ax.set_title('Overall Sustainability Index by Field (Higher = Better)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sustainability_index.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Key Findings - Extended

In [ ]:
best_sust = df_combined.iloc[0]
print("=== EXTENDED SUSTAINABILITY SUMMARY ===\n")
print(f"Most sustainable field: {best_sust['field_id']}")
print(f"  - Soil Health Score: {best_sust['soil_health_score']:.3f}")
print(f"  - Erosion Risk Score: {best_sust['erosion_risk_score']:.3f}")
print(f"  - Carbon Storage Potential: {best_sust['carbon_storage']:.2f}")
print(f"  - Overall Sustainability Index: {best_sust['sustainability_index']:.3f}\n")

high_erosion = df_erosion[df_erosion['erosion_risk_score'] > 0.5][['field_id', 'slope_pct', 'erosion_risk_score']]
print(f"Fields with HIGH erosion risk (score > 0.5): {len(high_erosion)}")
if len(high_erosion) > 0:
    print(high_erosion.to_string(index=False))

## 13. OM % Comparison: Top vs Bottom 10 Sustainable Fields

Comparing Organic Matter % between fields with highest and lowest sustainability indexes.

In [ ]:
df_combined = df_combined.merge(df[['field_id', 'avg_om_pct']], on='field_id', how='left')
df_combined = df_combined.sort_values('sustainability_index', ascending=False)

n_top = 10
df_top = df_combined.head(n_top).copy()
df_top['category'] = f'Top {n_top}'
df_bottom = df_combined.tail(n_top).copy()
df_bottom['category'] = f'Bottom {n_top}'

df_compare = pd.concat([df_top, df_bottom], ignore_index=True)

print(f"Top {n_top} sustainable fields - Mean OM: {df_top['avg_om_pct'].mean():.2f}%")
print(f"Bottom {n_top} sustainable fields - Mean OM: {df_bottom['avg_om_pct'].mean():.2f}%")
df_compare[['field_id', 'category', 'sustainability_index', 'avg_om_pct']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_cat = {'Top 10': 'forestgreen', 'Bottom 10': 'coral'}

ax1 = axes[0]
df_compare.boxplot(column='avg_om_pct', by='category', ax=ax1, 
                  patch_artist=True,
                  boxprops=dict(facecolor='lightgreen'))
ax1.set_title('OM % Distribution: Top vs Bottom 10 Sustainable Fields')
ax1.set_xlabel('Category')
ax1.set_ylabel('Organic Matter (%)')
plt.suptitle('')

ax2 = axes[1]
x = np.arange(len(df_compare))
width = 0.35
bars1 = ax2.bar(x[:n_top] - width/2, df_top['avg_om_pct'], width, label=f'Top {n_top}', color='forestgreen', alpha=0.8)
bars2 = ax2.bar(x[n_top:] + width/2, df_bottom['avg_om_pct'], width, label=f'Bottom {n_top}', color='coral', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(df_compare['field_id'], rotation=45, ha='right', fontsize=8)
ax2.set_xlabel('Field ID')
ax2.set_ylabel('Organic Matter (%)')
ax2.set_title('OM % by Field: Top vs Bottom 10 Sustainable')
ax2.legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'om_comparison_top_bottom.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Spatial Distribution of Soil Drainage Classes

Mapping field locations color-coded by drainage class from NRCS data.

In [ ]:
import json
import shapely

with open(DATA_DIR / "NC_field_boundaries_EPSG4326_2026-04-01.geojson") as f:
    geo_data = json.load(f)

centroids = []
for feat in geo_data['features']:
    field_id = feat['properties'].get('field_id')
    if field_id:
        geom = shapely.from_geojson(json.dumps(feat['geometry']))
        centroid = shapely.get_coordinates(shapely.centroid(geom))[0]
        centroids.append({'field_id': field_id, 'lon': centroid[0], 'lat': centroid[1]})

df_centroids = pd.DataFrame(centroids)
df_spatial = df.merge(df_centroids, on='field_id', how='left')
df_spatial = df_spatial.dropna(subset=['lat', 'lon', 'drainage_class'])

drainage_colors = {
    'Well drained': 'forestgreen',
    'Moderately well drained': 'gold',
    'Somewhat poorly drained': 'firebrick',
}
df_spatial['color'] = df_spatial['drainage_class'].map(drainage_colors)

print("Fields with drainage class data:", len(df_spatial))
df_spatial[['field_id', 'drainage_class', 'lat', 'lon']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for drain_class, color in drainage_colors.items():
    subset = df_spatial[df_spatial['drainage_class'] == drain_class]
    if len(subset) > 0:
        ax.scatter(subset['lon'], subset['lat'], c=color, s=150, label=drain_class, edgecolor='black', alpha=0.8)
        for _, row in subset.iterrows():
            ax.annotate(row['field_id'].replace('osm-', ''), (row['lon'], row['lat']), 
                       fontsize=7, ha='left', va='bottom', xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Spatial Distribution of Soil Drainage Classes\n(NC Field Cluster)')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'drainage_class_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Summary Statistics by Drainage Class

In [ ]:
df_drainage_summary = df_spatial.groupby('drainage_class').agg({
    'avg_om_pct': 'mean',
    'avg_ph': 'mean',
    'avg_cec': 'mean',
    'field_id': 'count'
}).rename(columns={'field_id': 'n_fields'})
print("Summary by Drainage Class:\n")
df_drainage_summary.round(2)